# M26 · RL foundations

_Curriculum · Domain 5 · Reinforcement learning_

**State an MDP, compute values, and know when full RL is worth the complexity.**

We build a tiny pacing MDP and run value iteration, then compute one REINFORCE policy-gradient term.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(26)

## MDP pieces

An MDP is $(\mathcal{S},\mathcal{A},P,R,\gamma)$. The Bellman optimality update is

$$V_{new}(s)=\max_a \sum_{s'} P(s'|s,a)[R(s,a,s')+\gamma V(s')].$$

In [ ]:
states = np.array(["under", "on_track", "over"])
actions = np.array(["bid_low", "bid_high"])
gamma = 0.9

T = np.array([
    [[0.70, 0.30, 0.00], [0.20, 0.70, 0.10], [0.00, 0.30, 0.70]],
    [[0.25, 0.65, 0.10], [0.05, 0.55, 0.40], [0.00, 0.10, 0.90]],
])

R = np.array([
    [[0.2, 0.7, 0.0], [0.3, 0.8, 0.1], [0.0, 0.4, -0.2]],
    [[0.1, 1.0, 0.2], [0.2, 0.9, -0.1], [0.0, 0.2, -0.5]],
])

assert np.allclose(T.sum(axis=2), 1.0)

## One Bellman sweep

Start with $V_0(s)=0$. The first sweep only sees immediate expected reward, because the future value term is zero.

In [ ]:
V = np.zeros(3)
q_values = np.sum(T * (R + gamma * V), axis=2)
V_one = q_values.max(axis=0)
best_actions = actions[q_values.argmax(axis=0)]

print(pd.DataFrame({"state": states, "V1": V_one, "best_action": best_actions}))
assert V_one[2] < 0.0

## Value iteration

Repeating the Bellman update propagates future consequences backward through the states.

In [ ]:
history = [V.copy()]
for k in range(20):
    q_values = np.sum(T * (R + gamma * V), axis=2)
    V = q_values.max(axis=0)
    history.append(V.copy())

policy = actions[q_values.argmax(axis=0)]
print(np.round(V, 3))
print(policy)
assert V[1] > V[2]

## Bandit or full RL

If today's creative choice does not change tomorrow's state, use a bandit. If spending now changes future budget state, pacing, or eligibility, the sequential state matters and an MDP is the right abstraction.

In [ ]:
bandit_rows = pd.DataFrame({
    "problem": ["variant pick", "budget pacing"],
    "state_changes": [False, True],
    "tool": ["bandit", "MDP or RL"],
})

print(bandit_rows)
assert bandit_rows.loc[1, "tool"] == "MDP or RL"

## One REINFORCE term

Policy gradients use sampled returns. For a softmax policy, the update direction is proportional to $\nabla \log \pi(a|s)G$.

In [ ]:
logits = np.array([0.2, -0.1])
exp_logits = np.exp(logits - logits.max())
probs = exp_logits / exp_logits.sum()
action = 1
return_G = 2.5
one_hot = np.array([0.0, 1.0])
grad_log_prob = one_hot - probs
grad_estimate = grad_log_prob * return_G

print("probs", np.round(probs, 3))
print("gradient estimate", np.round(grad_estimate, 3))
assert np.isclose(grad_estimate.sum(), 0.0)

## Plot value convergence

The values stabilize as the Bellman updates converge.

In [ ]:
hist = np.array(history)
fig, ax = plt.subplots(figsize=(6, 3))
for idx, state in enumerate(states):
    ax.plot(hist[:, idx], label=state)
ax.set_xlabel("iteration")
ax.set_ylabel("value")
ax.set_title("value iteration")
ax.legend()
plt.show()